# Data Description
- Background: 兩市S1來源蒐集、官方帳號歸屬線索與存取測試。來源目錄包含成功及失敗，不能當作全部已讀或全部核實。
- date received: 20261110
- Path to data file: 實際網址及抓取時間逐筆保存於來源JSON。
- Unit of observation: 每一URL抓取一次；manifest一列一個捕捉檔。review_decisions為人工／AI全文讀取後的有限判讀。
- Sample period: 2026-11-10來源快照；擬回溯2026-07-01至當日，並不表示快照包含此期間全部貼文。
- Known issues: 150份抓取紀錄、30次失敗。原HTML保留repo外私存，repo存完整伺服器可見文字、所有a連結、原始hash；非完整JS渲染。9份社群目標只有頁名／登入框。目錄頁含研究樣本外人物，未納入候選人資料。網頁內容不改寫候選人G欄。
- Definition for each variable: requested_url／final_url=要求與回應URL；fetched_at_utc=實際抓取UTC；http_status／error=回應或失敗；title／full_visible_text／links=解析之完整可見來源與連結；original_sha256／private_original=repo外原HTML查核線索；capture_id=URL指紋；manifest path／sha256=repo內來源JSON版本。review_decisions之reviews只支持帳號歸屬，target_reviews記錄9次內容不可計數測試；window欄保存使用者範圍及7月1日起的作業解讀。
- Data path: `data/raw/s1_tk_accounts_20261110/capture_manifest.json`及其中每一精確檔案；另含review_decisions.json與首次manifest.json。
- Role: raw / web representation and review decisions
- Verification status: 本文件clean-kernel讀取全部JSON、逐檔hash及捕捉數驗證；不認定所有文章已全文判讀。
- SHA-256: 逐檔固定於下方PINS；不是目錄hash。
- Next processing notebook: [S1工作單producer](../code/prepare_s1_tk_accounts_20261110.ipynb)


In [1]:
from pathlib import Path
import hashlib,json,re
from urllib.parse import urlsplit,parse_qs,urlencode
import pandas as pd
from IPython.display import display
for ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (ROOT/'AGENTS.md').is_file() and (ROOT/'templates').is_dir(): break
else: raise FileNotFoundError('請從專案目錄執行')
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def bound(rel,h):
    p=(ROOT/rel).resolve()
    assert p.is_relative_to((ROOT/'data').resolve()) and p.is_file(),rel
    assert sha(p)==h, f'輸入版本改變：{rel}'
    return p
def canon(u):
    u=u.strip().replace('\u2028',''); p=urlsplit(u); host=(p.hostname or '').lower().removeprefix('www.')
    if host in {'threads.net','threads.com'}: host='threads.com'
    if host not in {'facebook.com','instagram.com','threads.com'}: return ''
    path=p.path.rstrip('/')
    if not path or any(x in path.lower() for x in ['/sharer','/share/','/posts/','/videos/','/watch','/reel/','/p/','/stories/']): return ''
    q=parse_qs(p.query)
    query='?'+urlencode({'id':q['id'][0]}) if path.lower()=='/profile.php' and 'id' in q else ''
    return 'https://www.'+host+path+query
def platform(u):
    h=urlsplit(u).hostname or ''
    return 'Facebook' if 'facebook' in h else 'Instagram' if 'instagram' in h else 'Threads'


## Change for Your Own Data

固定明列所有來源；忽略macOS ._中繼檔。

In [2]:
PINS={'data/raw/s1_tk_accounts_20261110/0016841b30f0ef8c9688.json': 'eef22187c037853e7d0927a123dc25612976b22b955189cbadb169a1afb1cb10', 'data/raw/s1_tk_accounts_20261110/033d99a8d5ffc90d2955.json': '55085548e2d4f896a8c92b08d1c7ed44e0ed27a80428500456bb6f4b1e0dad6e', 'data/raw/s1_tk_accounts_20261110/058b1362bec1f063b8ae.json': '95e2f9c624316f764f5042531ee4fafd5755c8c1bbc077412f4b906ad29db775', 'data/raw/s1_tk_accounts_20261110/065ad17ea7d1dce9ba1b.json': '4a27754ac34cb5613da5aee2b93b1608cbd5af1783f3c08aa545412d62d8d008', 'data/raw/s1_tk_accounts_20261110/069a53d0e6eef35a9aa3.json': 'a4bc108d917852301316ae860d5ee24f20451d35ecdf6fde47fc8a3ae4aa9f12', 'data/raw/s1_tk_accounts_20261110/07542133412f56f91ad6.json': 'c5ac3f3ff3fcb94377dcccaadb065a682453ee010b5366f5bf948afb941ee736', 'data/raw/s1_tk_accounts_20261110/07a26baf7e1b46ad3094.json': '88501216cab0c347da5bf84a328e08b99ddab9672a4fd69548a0a8b57f343487', 'data/raw/s1_tk_accounts_20261110/0810d5febc033d5eb12e.json': '7634c8682564f3e24a231a7d9f477ba6ce214fcc7a22c5cc548045e3df4e3c60', 'data/raw/s1_tk_accounts_20261110/084b43753a98a110f997.json': '9b947c00d9754cd605753144e632824dc690fb0c81e08c7d3ebf0b8b0deb28c0', 'data/raw/s1_tk_accounts_20261110/0914bd34bf0b52ef2c4a.json': 'a80368f1672219343f065c08e7948e2b3532989f8fcb1824a93a55833e4e6586', 'data/raw/s1_tk_accounts_20261110/091d31bf4e24361657b6.json': '6754c5600140606a21b7045adfecae5afe1c15b9713825e7e4fce98c1e768713', 'data/raw/s1_tk_accounts_20261110/0a3c61e17821ee90df2b.json': '6a2aec453318cd15f7d29e7bfd4b0cb74b8afca5683b0498b1deeadb7d60b758', 'data/raw/s1_tk_accounts_20261110/0d14c1366543f202d58f.json': '1b296d13004f93731263f1f0a7318558a814d9c53203319f15cd06747f76f709', 'data/raw/s1_tk_accounts_20261110/119f6dc3d5c009beb683.json': '9745dbf25b3d858aaa3bae69f7c84c21e441434064fc91d3e13d274c78caa0c9', 'data/raw/s1_tk_accounts_20261110/11b34036ae1cf2a6d2fb.json': '51900914ef338c86b9dfe1737d6e992cde74d613bb8d1966ee6cfd08db9108d5', 'data/raw/s1_tk_accounts_20261110/160322320f1d9e9ca493.json': 'c6c0666bceb0fe2e668286bf7b714cf65493256bec228158940346fe8cc6b82c', 'data/raw/s1_tk_accounts_20261110/16ff52d840b184493bad.json': 'ba394bee6b9a7551e0873bb47599cfaaa8bf3a6463a439d1b5080270da90d9ec', 'data/raw/s1_tk_accounts_20261110/17b2034bab92bbc01981.json': '2eb9331c8aa3baed80aba38282489a96cc937786e07378d862334d7475edfde8', 'data/raw/s1_tk_accounts_20261110/188b9cf3de25f1907096.json': '7f1365f93c57dcd848cc14f388d4b2df2de6003b985b3b3afcc46612a9cfd17e', 'data/raw/s1_tk_accounts_20261110/18b645d57c360f773f2a.json': '8e847bb0cc95bc78a2c24636c2812e2ddd1b4bad69b66951a98e42d588a5df7c', 'data/raw/s1_tk_accounts_20261110/1e6d4fd8065f2f501447.json': 'bc8cb72402699340f455da8664cd0564f55a318f91032eaf6f5d702b3a2954cc', 'data/raw/s1_tk_accounts_20261110/1f2b1d7f37365f247dd8.json': '12079285e8b5f0ec82e87ba9b8c130728f486cba35fb4bf2acf889a2b7a267f7', 'data/raw/s1_tk_accounts_20261110/213da3b22a59ca8b5e68.json': '64e6dc1d18d105a7a4377c1076667221cf015f041f0a8d56878be7e812ef9881', 'data/raw/s1_tk_accounts_20261110/23b9b37dda15898daee2.json': 'eea7601800b20be0659a12265182ee1a243a36cee6361fa4bd1ab4d9ced5c857', 'data/raw/s1_tk_accounts_20261110/23e7d809bfc25fecfca6.json': '285e9c98692f2ea567d3eecd1e06c2353f9e371f7f7ee416b95db871a78d7e77', 'data/raw/s1_tk_accounts_20261110/28083438f9a4b1736331.json': 'c959e2eafdd3a920a785cd2123aca78d2769894563082f6e315427dc86f9214f', 'data/raw/s1_tk_accounts_20261110/28af5e49fdda388a628a.json': '13fcbe67dfb9f576e33e8af08e2c4438ed725d8528bda075f0265ae0741142f1', 'data/raw/s1_tk_accounts_20261110/298399a66c87c763ae87.json': 'f167d1117dc0fb4cab9421592a08ee15eccaa29dd1839ad7e4f500c8faab7245', 'data/raw/s1_tk_accounts_20261110/2a5185532403b98dc216.json': 'd954d6c83a7de9e9c172168a0b4d9a65976bc9c5a24e02cfe8b1aab0e6667ee4', 'data/raw/s1_tk_accounts_20261110/2a6b1a71104262ee1f4e.json': '29777d9e91e1b11ff95153098042579d0e0a6e140f04fe90c40bad504e171c64', 'data/raw/s1_tk_accounts_20261110/2aaad0dec5960720eda5.json': '3ef610543367682bd8c6e1dfdc9f9557cbc9284deb46d9da28496cd98e73c1cf', 'data/raw/s1_tk_accounts_20261110/2b7fb04f61c6b1727f31.json': '19cf059e232e692a70c49528346e3df064704afe1027e3c524ee351f6597202e', 'data/raw/s1_tk_accounts_20261110/2cc17bdc980712784e3b.json': 'c59fd61b55607145c5bb842d28f522fc5ca83e5f08117502e94abaad9ed3449e', 'data/raw/s1_tk_accounts_20261110/2eb7a8e23c1d7bc881c3.json': '1fc13aa682c2c436ec40ed9702a904ca1462ca4d0c745ef4359fa55d11d03e5c', 'data/raw/s1_tk_accounts_20261110/305a51c4b873c327011a.json': '33df926123fb4187a24fa3387ae03c39255d4e714d7071743600739ecc61ea6c', 'data/raw/s1_tk_accounts_20261110/327f96c4ad62c7e34ddc.json': '854b3106586912daa54b2cef07793faf0d647d02ab841d267370cc100baf14a8', 'data/raw/s1_tk_accounts_20261110/3363703d778e38c0204a.json': 'ca9d55ff2b4a15ce40ed05200274fb1e813def9e20ecaad283f746a7909180f3', 'data/raw/s1_tk_accounts_20261110/3720bded9b5f19546b49.json': '449d4cc4e17ad46f37fad04b3f3d0658961f13e783de914d86935a3f8b0d0080', 'data/raw/s1_tk_accounts_20261110/37de4032897b43460261.json': 'f2972d0b7310e93de09fefebf1da6bbaa006d9f99cd8c00ab41055b4b2f3831a', 'data/raw/s1_tk_accounts_20261110/38c8a89a30712044dac9.json': '2ef4a0e41019cb672ecc93dfd982de321edd098dd59a9a1387025a7efec445d9', 'data/raw/s1_tk_accounts_20261110/3a8164b31c82e621ab59.json': '2691a208d4af7d80f2114e117050ff29900e9cc394a118220115851859c9c444', 'data/raw/s1_tk_accounts_20261110/3c5b051ca3b5f6797d8c.json': 'e558ec546839a473e50bd0345b4088ad75f1c8414e0c9505d957fa9a34c4aa08', 'data/raw/s1_tk_accounts_20261110/3d73dc53b62d36824db0.json': '2db2ee0805360ee9c65783c8cdebc5d8a59291afa92e21e3410b7f2a7b282b86', 'data/raw/s1_tk_accounts_20261110/3dad9737fe02bf5b9822.json': '7ebab7322f224ec2b73ec4d80a0bef4013bd4976c47efcce73c9e64fd73d808d', 'data/raw/s1_tk_accounts_20261110/3daf6654f941bdf47ebd.json': 'b804399b78dfc30b3ea1d34152c1d4eb41f70d39d4e2d76e4ed6744a1a07cc08', 'data/raw/s1_tk_accounts_20261110/3f5cebd151ca89611782.json': '10d8b7e4c36bb99473ed044ca94b3352f67e56f95a182a16dae07bc1c82c538f', 'data/raw/s1_tk_accounts_20261110/3ff97a9eb3a256861259.json': '6a72da84febf0dc90e926b26e1efdae98e3aef31cf34d49b4eaf7821eeca0340', 'data/raw/s1_tk_accounts_20261110/42dfce12a0cf184f0618.json': '873b3a413434f41e3b5cf07ae593f0dd56486352b81845e5eb3bd6a51e163a5a', 'data/raw/s1_tk_accounts_20261110/4531c52fae9e39d2c459.json': '20490d512d61688f270b4d50282317336c7a0c8d6cfaeabe247771150028ce66', 'data/raw/s1_tk_accounts_20261110/46b1497bf83267751c3f.json': '99fbe274eede8bc2972e7174c0e0799b59bd4b21251ae647273e8de7769db26c', 'data/raw/s1_tk_accounts_20261110/4829d7f92a5ce7b4c386.json': 'f1fccaabcf78e9102b17dc4bb87d479df6adb41dbbaafbeb354918676e048896', 'data/raw/s1_tk_accounts_20261110/4981508a01bb0fc08fb7.json': 'e6586adc5448f8d619aa26d42cfc67ffc820d5763c368613aef26af9240bdc40', 'data/raw/s1_tk_accounts_20261110/4b68edf3fc599a2eba40.json': '4c04559fdf01a10146c105d726a612b95337918b0ead0e9503f2da168b26af35', 'data/raw/s1_tk_accounts_20261110/4d9b5784d76cd56b15de.json': '78a37363a8deac05a65ed68bbcb5e2a5e764ed353365e3031f598e93b9d8ebf6', 'data/raw/s1_tk_accounts_20261110/4dc8f7c95636df508b65.json': '1e78453a1a630932171a29b91a18450d48e2004544e59057ca720a807e1abc25', 'data/raw/s1_tk_accounts_20261110/4e448763a345c3a30c55.json': 'f33935b01818d48d90c3cdf1ad58eefd9b54623b47ec848f7eba7f57c2078c0f', 'data/raw/s1_tk_accounts_20261110/5024b777db3dc15d9b66.json': '958674dc2c2d4b3f01940f364d9fda70405f824f800c30fae5baec93cf60f8b8', 'data/raw/s1_tk_accounts_20261110/509e7587027259955bdc.json': '8639e8c03a8e7d3e7eefb17f0177b55553876fcfd57ee63252177635ec284cd2', 'data/raw/s1_tk_accounts_20261110/50b22ce8b7a3e6fd7409.json': 'c84aa58f2360c0d57e160ee3c0f8b63ce9b34a747e25cdd65e5dd555adf793e2', 'data/raw/s1_tk_accounts_20261110/50b5b4e6cf6ba5c9c04e.json': '7abbb4a5975f6478244badffbd07875142d94bf2abb2ee6f0dbe8be3f9855cbb', 'data/raw/s1_tk_accounts_20261110/524f9f72447ce1a08c59.json': 'a6d41121f1c80181691d8aaf2c16ab0360cbb9a70303bce92c766ceacc82fc44', 'data/raw/s1_tk_accounts_20261110/52d517f2cd091856f373.json': '257a985c858f78c0935aa704d2044b96a20cb84d3f3a47755636b69f2d7d7c99', 'data/raw/s1_tk_accounts_20261110/548156f262a50ae0ad22.json': '08768de8f24929a1b7fc0bb324d4e369c8d4e2f355ea2d7fdd240eba0856126f', 'data/raw/s1_tk_accounts_20261110/589dfeca10a8d21df023.json': '6c3c29f61d4426f59c83e03d0bd28182cbcf55d1d1e4b21e0f192a23bf2a77fd', 'data/raw/s1_tk_accounts_20261110/5c6daa4c5d03c1e93c64.json': 'bf8ba6547b3716cf10397af5af86fb7b271d997473d6e1dbf1bf241c8530947f', 'data/raw/s1_tk_accounts_20261110/5de6857863f18dc55226.json': 'd420e42eef7357333009547a3105b3129fed6502c3154b8376efb5c5a2112991', 'data/raw/s1_tk_accounts_20261110/63c97af6ef12bda56f13.json': '23f4c1a46596eed35cb607f817031986eee7d4569c930e9c06264418630878a9', 'data/raw/s1_tk_accounts_20261110/6683731d475f858485ba.json': '95855ff50390911d91f53689c0d10e1b2060c8b7a0bd0081f30e994981018176', 'data/raw/s1_tk_accounts_20261110/66d6e6aa9e950bc4eae1.json': '55310db42722f4936e890311f94925c3a3c05c57e893b2fe3d5e7e76f14b6d92', 'data/raw/s1_tk_accounts_20261110/6b1a7b2f74b25783e1da.json': '014172a3fa4723699996aa3b55108574a06f8ce61a876dafdf2977ee8c2162db', 'data/raw/s1_tk_accounts_20261110/6b48718b731b217b3a07.json': '6cbcd301897cc1ace64c89a138ca8c4685586d0b853e76d5d48cf2fa66289dac', 'data/raw/s1_tk_accounts_20261110/6d70f296f13d0d4fc261.json': '787181b7c3c9f845ce6d31dc8043280541c01a88ba8c0baaffe1eb482c87f285', 'data/raw/s1_tk_accounts_20261110/732fa3dbed00538214c6.json': '2484f3b5f6c4c7baf21dabac53bc20c3a558483be553d441fc970a380d279e41', 'data/raw/s1_tk_accounts_20261110/765538154bfbe063ad2b.json': '29cf062441f766259f752a6933654c7efa8d5323b8446916e69f4f0e9ded5d0a', 'data/raw/s1_tk_accounts_20261110/77018305524885cae88e.json': '5e012c0f1e36c4cd6439cdaa90d3c9d8293292e19715c3d6ba18d8896f1ed63f', 'data/raw/s1_tk_accounts_20261110/786cd3f6c751211897eb.json': '0316a18f0b4d4fb6d1442e3bc57136722f42d98f02d292e2f11dae6d152d5504', 'data/raw/s1_tk_accounts_20261110/791ed83075ae33efd279.json': '64a354480630486e0ac22821e36d1499571365735d7e4881bc0dba5bb6f286e6', 'data/raw/s1_tk_accounts_20261110/79f6cf9d59735892f18b.json': '2176066bfe5524caacb466b9800775f7c96933e353f0dc76ed2908bf32ea0642', 'data/raw/s1_tk_accounts_20261110/7cb46ef052b660363b98.json': '284c76955825631cf5a291d24afa225023291d84c133fb70ccfc9436971cfa2e', 'data/raw/s1_tk_accounts_20261110/7d66040d519488700da0.json': 'd48698866072b859716fccf9944e579ae6d11701afea68368825bd73a07fa414', 'data/raw/s1_tk_accounts_20261110/7eb2185a391a72b84e56.json': 'e317bee32a7d8816fdf46b91c29a8d359ae86c015a2486c3b79528110ee7ccbb', 'data/raw/s1_tk_accounts_20261110/8104f13af379a13ec2f6.json': '431b24ef65861ad968484863235c51b55603fad486d9395a2690cb67e4d2ea51', 'data/raw/s1_tk_accounts_20261110/86666c9a4a9d76265813.json': '87679b13e07981564674d106cf9755a5cc09a3b1e65dea20f49c73b656ab1c27', 'data/raw/s1_tk_accounts_20261110/8679e1b01d12659061c2.json': 'e8f7a53e51a055051fc68d2d78cf98cff3533e48555546577927b7b716fc86d7', 'data/raw/s1_tk_accounts_20261110/876c55f9b2be605b962b.json': '6645d6ce99b40b2608da7875bec7116c58509a5becad2615454c674ecff76803', 'data/raw/s1_tk_accounts_20261110/87c6b862fd1b603828fb.json': '2fca330c7c21cc1e8af790d3c6a4b27146a38097a1b675123328823c6a23af08', 'data/raw/s1_tk_accounts_20261110/8853da31089f05ed88af.json': '2b0536db4ad694a2f553514ca81b5bdf8307466f88e853d75cb6e1316e587dae', 'data/raw/s1_tk_accounts_20261110/8ce6581fdcc1d0f3eaa3.json': 'dd6dc41a8d60cbf40828f92a28579f9dd44fbc27126544154a2d4d93b2a007b6', 'data/raw/s1_tk_accounts_20261110/8d7bf151a0628564738e.json': '0c88cc8324c76aa9e5ed463b70ac155c2f13ac29d7877ed64c216aee65dfb8bf', 'data/raw/s1_tk_accounts_20261110/8ebca21517f7ac7b0261.json': '1e63e9fd845e9256eac18e756c955a9b66035e52a5f1dd4307b00aaa5080fa4c', 'data/raw/s1_tk_accounts_20261110/8f7f8e430b81d4096240.json': '26a56787f5427b427b04b6476dbbdba8ecebb8aebfba519b500f2afe354add67', 'data/raw/s1_tk_accounts_20261110/9084a09e3b4dafe2bdab.json': '62edc751dfe18dc414c9366d29761e017a9d5c0ccb4ba33acc2769ddaf964a7c', 'data/raw/s1_tk_accounts_20261110/9848df7f172361b07bea.json': '9865b29d84305883179d56140d7690c587489e923f30553cb0ce640605d58d98', 'data/raw/s1_tk_accounts_20261110/a25f32bc5bc915c6154f.json': '7e7fe4fc9aeb324592e618dcc842b68bc446728b67643058cc425fbdfe6c3190', 'data/raw/s1_tk_accounts_20261110/a46722aa46f47380a652.json': '7172d8f57bf2785796c9788606832a67b692f6f668a9230872a0dbf552019e51', 'data/raw/s1_tk_accounts_20261110/a7f02ca94fccf52d2806.json': 'cd88e6783e696c23dfa6154411a77f1dcf590efee460b01d028d24f3ae4201aa', 'data/raw/s1_tk_accounts_20261110/abb4b4790b1a8d2d7098.json': 'f90b2e9ab3b3bbaffed95fb4beb78054dbfb5a163d620a8750edc28505ba50dd', 'data/raw/s1_tk_accounts_20261110/ac2033c7247cce26be44.json': 'c65cec4969a88ce82ccfebce646d22a2cfb0e58c49fd005c298fd764fbe42f1e', 'data/raw/s1_tk_accounts_20261110/ad7a2ba912257825374d.json': '62dd3f88ce03341550dfc4962e1ee1c3588290b49219a77d3a8669b1672b7360', 'data/raw/s1_tk_accounts_20261110/ae01a9713640eba95cbd.json': 'fc9d294d62f0185ab6a046a4c53b985f8c8e48a860f024866e7539e4ca8cb24e', 'data/raw/s1_tk_accounts_20261110/ae1bc8263b803a57f629.json': '86aeb7bd3e29a237a27fdbf2daf2e6a80e56696f00a3ce4149c4e103133cc9f6', 'data/raw/s1_tk_accounts_20261110/b32549be23bf991f708f.json': '4499643e6342dc633588d40bb357a7967d384dc2383eb9d16ff92f562a8c9c7d', 'data/raw/s1_tk_accounts_20261110/b3ae42b4078b4bc7b385.json': 'd3272f3608aeaafa045efbadf2530314bfdb7253d13d4b4c445a32956ffbe504', 'data/raw/s1_tk_accounts_20261110/b3d3888092fc612cb5b9.json': '39fbf4ebd912b794fe1c982a12c682d09144971795ae50854616117dd3b94bc4', 'data/raw/s1_tk_accounts_20261110/b5569e490b3760cbe5db.json': 'c547b1b1ed389ba3614d226a58efd2969b7af5376768c202173f89dcab726637', 'data/raw/s1_tk_accounts_20261110/b75ee01e801c812679a8.json': '5fcd73cf8c0229d6976620967b93b77e9f4c18b4f57f780f90db4d59e7a64b19', 'data/raw/s1_tk_accounts_20261110/b7cd7413d69d45ea189d.json': '52c9afee9648b3543bfbc9dff35f39264519a8f0ccf82749f1732083770f57b6', 'data/raw/s1_tk_accounts_20261110/b8a65fd518df179d8524.json': 'c3e5c77ab68cf32e71ff84a1c6efc863f51568371209a8247b6eace24e1dc823', 'data/raw/s1_tk_accounts_20261110/b9b087480c8a7a682cca.json': '13e2483da3929a922bba809b3d912a5850a57103af7ec2ef5fc98f92421dd95c', 'data/raw/s1_tk_accounts_20261110/ba5c7302d507e102d6d7.json': '0c12e0714946201baba7c0d3b76d14febe575008d256d8f06b63cb36f504ca68', 'data/raw/s1_tk_accounts_20261110/ba94d437d730beb82de7.json': '6df2190f51ee7d78af3fd5e7cfa510511af5f27af33f2274e42cdd2e37458989', 'data/raw/s1_tk_accounts_20261110/bd53782937724b04fa5f.json': '0f6a667a16289061fa54f05e5a8a8a0396f9a5f6aa6645a20a775ba48035ccd4', 'data/raw/s1_tk_accounts_20261110/be0f523016e881f3a48f.json': '86f3ff7b82c25d81458f4deb147ec4cfbdacf2adfb13cf1202034f48582c3d5f', 'data/raw/s1_tk_accounts_20261110/be731789ee7f56fc55d9.json': '6ea4e650eb0492a77fd22f103b9dcf7c9f33db0ae781ada4715d772176e6556c', 'data/raw/s1_tk_accounts_20261110/bee07d8877763f80e05a.json': '6bc7a4fb7cd770d442791e5172c2dde67e2af21c85ec4f10dcd712b455f49259', 'data/raw/s1_tk_accounts_20261110/c4da2a38f26c6709b900.json': '5177751c6aa5bef20beaf777983b95d7f39cf03888736350934fdeeb6ff623e2', 'data/raw/s1_tk_accounts_20261110/c53c81ba6de4a384ba3a.json': 'dabc6bba57c27870481f83f7fe3dd33d958fba8a533e0d0d5201fd42f812fb49', 'data/raw/s1_tk_accounts_20261110/c5e634a0c1f47a5dd03b.json': '3dd17d3218cc2c0337a59fab7cfc50da813eeea20e39dd0f2538f05f59862c9f', 'data/raw/s1_tk_accounts_20261110/c7cebfaa688ae4109ed3.json': '20b53db06a3e774ee6b57f63a1a22baa47bf08a9ae5d4f05029f675ee0eb53c7', 'data/raw/s1_tk_accounts_20261110/c9313cf336eb384a5491.json': 'da1ba5427e0662f272074de02b3be2abc543a0d07c90f1a385fc013de348a70b', 'data/raw/s1_tk_accounts_20261110/ca24a3eb1da1a498883c.json': 'a841df0ed0c21a3967a79db57d7eaefb1e48a6b4730d39dbe08ce8b53b4153b6', 'data/raw/s1_tk_accounts_20261110/cab91a155008312e422c.json': 'dab0114cc3ef9c58025cfeb1171672bb5e28a4fdb1ab48d5bcf14640b7b216c6', 'data/raw/s1_tk_accounts_20261110/cc0c1e1c2980eeb19dfd.json': '5c7dcf2c7e15811a43d242cececb1aa4da7fd69fb7199f8ca227b8d0d9a8f93b', 'data/raw/s1_tk_accounts_20261110/cc117f640a204a2f61e4.json': '0a51cbbfabdca44070f6bd21635b9b71275bfd3bace5e3a21aec62b7fe01c474', 'data/raw/s1_tk_accounts_20261110/ceeba98d3e523e616900.json': '3e9bd723cf7ca9069221d5385d9df704df76cc14ca58f9f2f7a534297fac7559', 'data/raw/s1_tk_accounts_20261110/ceeeeff6db283d350fda.json': 'a0102df472d9d8581452a724ae55760793037316b503c76b49516f130b63155e', 'data/raw/s1_tk_accounts_20261110/cef3d8b2984123575d1b.json': '239fc81159c72c35049836230c45e1e20cbf5848218228095c59a30441bb8abe', 'data/raw/s1_tk_accounts_20261110/cfef5ad787502b3b4253.json': 'b04bb761e3aac179d49c64dba40106baf5694af42b5859e27cef63165fdc2808', 'data/raw/s1_tk_accounts_20261110/d08b17c685eeeea3a02d.json': '171bda650c1b04ebc0f341d668cd3bdbc4c9d67309a1157d069fa81ef62a661a', 'data/raw/s1_tk_accounts_20261110/d0cd722b17d049f7b55b.json': '59e7375f3ed46f4b93a3a50b307422033e1f09cdf56f5e10097bfe5797325847', 'data/raw/s1_tk_accounts_20261110/d4417bc23839e47d4ace.json': '6c0b53f5b224409d251f5fd9663aa697bbc068a34b55616adb4362460f2b9153', 'data/raw/s1_tk_accounts_20261110/d488d97340ee5087a148.json': '1c56ab963d63d4859e7a7078367bda3516c8ad31641f40988da11b4553a109c3', 'data/raw/s1_tk_accounts_20261110/d82c80ff0e5186d24fb3.json': '4c80ef2070b5fee6a26efdc2b90ecd6c5c8e0aeca5dcc5d3727e41cd7597c0c0', 'data/raw/s1_tk_accounts_20261110/e5e00b5b3ca3572d97a9.json': 'a9d0a733e5bc048bfb5f1822187ec91343ad8839eb08029c03a82f24b17bf2fe', 'data/raw/s1_tk_accounts_20261110/e70063ce6deadaf5dafb.json': '8d932a5667d4254501579e6c0c7e7a259ae9ea5e4c3a93245bf4a385f1b3c25d', 'data/raw/s1_tk_accounts_20261110/e83cf474bc53c23554d5.json': '56e98e25048a3fdfb4ed7408c6ea226173ad45285cfa28418c4bc99df80041ec', 'data/raw/s1_tk_accounts_20261110/ea8701d95029e885aad6.json': '4545d220cd44c15b64386c1282a66f4d49b494d534092259bd8a1d12120b4d0c', 'data/raw/s1_tk_accounts_20261110/ecbc4b65e8098bc57ee0.json': '767b094bdd36cd77ddbb1c2a6638adba7d76f988e89d7d0078ce684a270dfa0f', 'data/raw/s1_tk_accounts_20261110/f00ad983eec98dd99dfc.json': 'b5d12c525da20071afcfc836ed2d9be80c6d77f705e73ac5bbb2a7cdd36a27ea', 'data/raw/s1_tk_accounts_20261110/f265629a7e8db7960646.json': '983027fdcda44a28d5175917f35e3645d63baca7f8fa9c65f913978c4c7aecfc', 'data/raw/s1_tk_accounts_20261110/f290ab192c25b5294a8b.json': '28ca88c8b58a8b5a426536d308e561516e77f7b01e71a384a366699804222181', 'data/raw/s1_tk_accounts_20261110/f56b001e4180b2b14338.json': 'f0e46ea793fa732bca693a781c7b3e51f3464654f2ffb9da866f433a152ac00b', 'data/raw/s1_tk_accounts_20261110/f681db580e269671952b.json': 'b08406f4db5146285d1a415be28283bbdb812057c379f0986ec8f2151c40766b', 'data/raw/s1_tk_accounts_20261110/f73d9ea0f8c2bb0bccbf.json': '1e09db2bbeaf0d707e5973ee119577d831fedcae724c4889d2556ec6c2582748', 'data/raw/s1_tk_accounts_20261110/f79310e215d0da9dcbb6.json': '0adda945a7643e8e8843e77c11cdad9df37e2163b008f5abde7ff643fa4eaecc', 'data/raw/s1_tk_accounts_20261110/f7da0284b7311171e40c.json': '623bafe395664b8eab3ba3db03703eadd81ab49e63fcb81afb0b8e1c995e8b0c', 'data/raw/s1_tk_accounts_20261110/f92ac537b9cf789ab439.json': '06cc9a7d99acbf550b4e350ae53251664481905577efd169c6b407ec12019bfe', 'data/raw/s1_tk_accounts_20261110/fa2a957f745d9a435d8f.json': '581d717461ce17ba6a1700db804a1f48e05b061247e3757ed4d778e6113435c0', 'data/raw/s1_tk_accounts_20261110/fa5fe41cea494b6578f7.json': '326da8a14611f464f484ed3e981b6e5be83efeaf62822f90e147a7b5545d09bc', 'data/raw/s1_tk_accounts_20261110/fc585f3aa38cb8e275b0.json': 'a0220f09aa7a6109beb54eb5930d8efb422b9b2c6138b024bcc91890c361462c', 'data/raw/s1_tk_accounts_20261110/manifest.json': '321cbbc1beefdf0b9bbb7de6c0b7e0b41e9e7cde3dd7f33f81f232daace927fc', 'data/raw/s1_tk_accounts_20261110/capture_manifest.json': '8d2daf779b127db37f2b9eec89c060c02e5961f0ec8fc9f3920bad92d9aabc68', 'data/raw/s1_tk_accounts_20261110/review_decisions.json': '7da602689ca76b93ce4187ac74117dc36c96e25f15f61542ccbc2622fbfc4839'}
raw={rel:json.loads(bound(rel,h).read_text()) for rel,h in PINS.items()}
manifest=raw["data/raw/s1_tk_accounts_20261110/capture_manifest.json"]
assert all(x["path"] in PINS and x["sha256"]==PINS[x["path"]] for x in manifest)
df=pd.DataFrame(manifest)

## Summary

In [3]:
print('來源檔',len(manifest),'失敗',df.error.ne('').sum()); print('原HTML在repo外，repo內為解析表示。')

來源檔 150 失敗 30
原HTML在repo外，repo內為解析表示。


### Sample

In [4]:
display(df.head(10))

,path,sha256,requested_url,http_status,error
0,data/raw/s1_tk_accounts_20261110/0016841b30f0e...,eef22187c037853e7d0927a123dc25612976b22b955189...,https://www.tcc.gov.tw/Councilor_Content.aspx?...,200,
1,data/raw/s1_tk_accounts_20261110/033d99a8d5ffc...,55085548e2d4f896a8c92b08d1c7ed44e0ed27a8042850...,https://www.kcc.gov.tw/MemberInfo_New.aspx?n=3...,,ReadTimeout: HTTPSConnectionPool(host='www.kcc...
2,data/raw/s1_tk_accounts_20261110/058b1362bec1f...,95e2f9c624316f764f5042531ee4fafd5755c8c1bbc077...,https://www.tcc.gov.tw/,200,
3,data/raw/s1_tk_accounts_20261110/065ad17ea7d1d...,4a27754ac34cb5613da5aee2b93b1608cbd5af1783f3c0...,https://www.kcc.gov.tw/MemberInfo_New.aspx?n=3...,200,
4,data/raw/s1_tk_accounts_20261110/069a53d0e6eef...,a4bc108d917852301316ae860d5ee24f20451d35ecdf6f...,https://www.kcc.gov.tw/MemberInfo_New.aspx?n=3...,,ReadTimeout: HTTPSConnectionPool(host='www.kcc...
5,data/raw/s1_tk_accounts_20261110/07542133412f5...,c5ac3f3ff3fcb94377dcccaadb065a682453ee010b5366...,https://www.tcc.gov.tw/Councilor_Content.aspx?...,200,
6,data/raw/s1_tk_accounts_20261110/07a26baf7e1b4...,88501216cab0c347da5bf84a328e08b99ddab9672a4fd6...,https://www.facebook.com/chiangwanan,200,
7,data/raw/s1_tk_accounts_20261110/0810d5febc033...,7634c8682564f3e24a231a7d9f477ba6ce214fcc7a22c5...,https://www.tcc.gov.tw/Councilor_Content.aspx?...,200,
8,data/raw/s1_tk_accounts_20261110/084b43753a98a...,9b947c00d9754cd605753144e632824dc690fb0c81e08c...,https://www.kcc.gov.tw/MemberInfo_New.aspx?n=3...,200,
9,data/raw/s1_tk_accounts_20261110/0914bd34bf0b5...,a80368f1672219343f065c08e7948e2b3532989f8fcb18...,https://www.pumashen.org/,200,


### Summary Stats

In [5]:
display(df.http_status.value_counts(dropna=False).rename('URL數').to_frame()); assert len(manifest)==150
assert df.path.is_unique
assert all(sha(ROOT/rel)==h for rel,h in PINS.items())
print('全部JSON、固定hash及清冊唯一鍵驗證通過。')

,URL數
http_status,
200,120
,30


全部JSON、固定hash及清冊唯一鍵驗證通過。
